<a href="https://colab.research.google.com/github/vorhersager/deep-learning-jax/blob/main/Tutorial_04_Convolutional_Neural_Networks_and_Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Tutorial 4: Building a "Humans vs Cars" Classifier using Transfer Learning


Instructor: John Sipple

### Overview

In this tutorial, we construct a Convolutional Neural Network (CNN) using the JAX ecosystem, specifically leveraging Flax for building neural network layers and Optax for optimization. The primary focus is exploring Transfer Learning—the technique of taking visual features learned on a base task and applying them to a new, related task. This approach simulates a common real-world scenario where limited data or compute necessitates leveraging a model's pre-learned visual understanding rather than training a new network entirely from scratch.

### Key Concepts Explored:

*
**The JAX/Flax Stack:** Understanding Flax's functional, stateless design where parameters are passed in and updated states are returned, which distinguishes it from object-oriented, stateful modules seen in other frameworks.


*
**CNN Architecture & Introspection:** Designing a model with a distinct Feature Extractor (Convolutional and Pooling layers) followed by a Classifier Head (Dense layers). The notebook includes a method to dynamically inspect the Flax parameter tree, deduce layer shapes, and automatically render a visual block diagram of the network's architecture.


*
**Base Model Training (Task A):** Training the CNN from scratch on the CIFAR-10 dataset to classify Frogs versus Ships, forcing the network to learn fundamental visual representations.


*
**Visualizing "Receptors":** Extracting specific parameters from the trained model to process a single test image through the first Convolutional layer. This allows you to visually plot the intermediate feature maps and observe how the network acts as an edge or color detector.


*
**Transfer Learning & Gradient Masking (Task B):** Repurposing the trained network to classify People versus Vehicles using the CIFAR-100 dataset. The tutorial walks through the exact mechanics of transfer learning: resetting the Dense layers, freezing the Convolutional layers by explicitly masking their gradients to zero during the backward pass, and exclusively retraining the new classification head.

# Introduction to the JAX Ecosystem

In this tutorial, we are using the **JAX** ecosystem. Unlike TensorFlow or PyTorch, which are often "monolithic" frameworks, JAX is modular. This means we assemble our deep learning stack from several specialized libraries.

Here is a breakdown of the three key libraries we will use:

### 1. **JAX** ( The Engine)

**JAX** is the foundation. It is a library for high-performance numerical computing.

* **Autograd:** It can automatically differentiate native Python and NumPy code.
* **XLA (Accelerated Linear Algebra):** It compiles your Python code to run efficiently on GPUs and TPUs.
* **JIT (Just-In-Time) Compilation:** We use `@jax.jit` to compile our training steps into super-fast XLA kernels.

### 2. **Flax** (The Neural Network Library)

**Flax** is a high-level neural network library built *on top* of JAX.

* It provides pre-built layers like `nn.Conv` and `nn.Dense`.
* **Philosophy:** Flax is explicitly designed to be functional and stateless. Unlike PyTorch `nn.Module`s which hold their own state (weights), Flax Modules are pure functions. You pass the parameters *in* and get the new state *out*.

### 3. **Optax** (The Optimizer Library)

**Optax** is a library dedicated solely to optimization.

* It provides optimizers like `Adam`, `SGD`, and `RMSProp`.
* **Composable:** You can chain transformations (e.g., `clip_by_global_norm` + `adam`).
* It also provides standard loss functions (like `softmax_cross_entropy`).

**Summary:** We define the model structure in **Flax**, calculate gradients using **JAX**, and update the weights using **Optax**.

#Overview
In this tutorial, we will explore Transfer Learning: the technique of taking features learned on one task and applying them to a new, related task. This is the foundation of modern computer vision (e.g., using a pre-trained ResNet).

##The Plan:

1. **Task A (The Base Model)**: Train a Convolutional Neural Network (CNN) from scratch to classify Frogs vs. Ships using the CIFAR-10 dataset.

2. **Task B (Transfer Learning)**: Use the trained feature extractor from Task A to build a classifier for Humans vs. Cars using the CIFAR-100 dataset. We will "freeze" the convolutional layers and only train the final classification head.

In [ ]:
# --- Step 1: Install & Import Dependencies ---
!pip install -q flax optax tensorflow-datasets

import jax
import jax.numpy as jnp
from jax import random, jit, grad, value_and_grad

# Flax (Neural Network Library for JAX)
from flax import linen as nn
from flax.training import train_state
from flax.core import unfreeze


# Optax (Optimization Library for JAX)
import optax

# Data Loading
import tensorflow_datasets as tfds
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import jax.numpy as jnp
from jax import random


# Ensure JAX is using GPU if available
print(f"JAX Devices: {jax.devices()}")

#Part 1: The Architecture & Data Pipeline


We define a standard **Convolutional Neural Network**. It consists of a **Feature Extractor** (Conv layers + Pooling) and a **Classifier Head** (Dense layers).

In Transfer Learning, we often keep the Feature Extractor and swap out the Head.

In [ ]:
# --- Step 2: Define the CNN Architecture ---

class CNN(nn.Module):
    """A simple CNN with a distinct feature extractor and classifier head."""

    @nn.compact
    def __call__(self, x, train=True):
        # --- Feature Extractor (The part we will transfer) ---
        x = nn.Conv(features=32, kernel_size=(3, 3))(x)
        x = nn.relu(x)
        x = nn.max_pool(x, window_shape=(2, 2), strides=(2, 2))

        x = nn.Conv(features=64, kernel_size=(3, 3))(x)
        x = nn.relu(x)
        x = nn.max_pool(x, window_shape=(2, 2), strides=(2, 2))

        x = nn.Conv(features=64, kernel_size=(3, 3))(x)
        x = nn.relu(x)

        # Flatten
        x = x.reshape((x.shape[0], -1))

        # --- Classifier Head (The part we will re-train) ---
        x = nn.Dense(features=64)(x)
        x = nn.relu(x)
        x = nn.Dense(features=2)(x) # Binary classification (Logits)
        return x

# --- Step 3: Data Helper Functions ---

def get_dataset(name, classes, split, batch_size=64):
    """
    Loads a dataset (CIFAR-10 or CIFAR-100), filters for specific classes,
    and remaps labels to 0 and 1.
    """
    ds_builder = tfds.builder(name)
    ds_builder.download_and_prepare()

    # Load raw data
    ds = ds_builder.as_dataset(split=split)

    # Filter function: keep only rows where label is in 'classes'
    def filter_fn(x):
        return tf.reduce_any(tf.equal(x['label'], classes))

    ds = ds.filter(filter_fn)

    # Map function: Normalize images and re-map labels to 0 or 1
    # classes[0] -> 0, classes[1] -> 1
    def map_fn(x):
        image = tf.cast(x['image'], tf.float32) / 255.0
        label = x['label']
        # Create binary label
        new_label = tf.where(label == classes[0], 0, 1)
        return image, new_label

    ds = ds.map(map_fn).shuffle(1000).batch(batch_size)
    return tfds.as_numpy(ds) # Convert to numpy for JAX

def show_batch(batch, title="Data Sample"):
    images, labels = batch
    fig, axes = plt.subplots(1, 8, figsize=(12, 2))
    for i, ax in enumerate(axes):
        ax.imshow(images[i])
        ax.set_title(f"Lbl: {labels[i]}")
        ax.axis('off')
    plt.suptitle(title)
    plt.show()

## Visualizing the Model Architecture


Understanding the flow of data through a neural network is essential for debugging and design. While we defined the CNN class in code, visualizing it as a graph helps confirm the layer sequence and dimensions.

The following function, `draw_cnn_architecture`, dynamically inspects your Flax model to generate a block diagram.

**How it works:**

1. **Model Introspection:** It initializes the model with a dummy input to create the Parameter Tree (variables). In Flax, parameters are stored in a nested dictionary (e.g., params['Conv_0']['kernel']).

2. **Shape Inference:** It iterates through this dictionary, inspecting the shapes of the kernel weights (filters) to deduce the layer properties:

>* **Conv Layers:** Kernel shape (3, 3, 3, 32) tells us we have 32 filters of size 3x3.

>* **Dense Layers:** Kernel shape (1024, 64) tells us we have 64 neurons connected to 1024 inputs.

3. **Dynamic Rendering:** It uses matplotlib.patches to draw a box for each discovered layer, scaling the diagram automatically based on the depth of your network.

**Why this matters**: This visualization confirms that our "Feature Extractor" (the blue Conv layers) sits before the "Classifier Head" (the green Dense layers), setting the stage for our Transfer Learning strategy where we will freeze the blue parts and retrain only the green parts.

In [ ]:

def draw_cnn_architecture(model_class, input_shape=(1, 32, 32, 3)):
    """
    Visualizes the architecture by inspecting the initialized parameter tree.
    """
    # 1. Initialize the model to get the parameter structure
    model = model_class()
    rng = random.PRNGKey(0)
    variables = model.init(rng, jnp.ones(input_shape))
    params = variables['params']

    # 2. Extract Layers from the parameter keys
    # Flax parameters are usually nested: params['Conv_0']['kernel'], etc.
    # We sort keys to ensure correct order (assuming names like Conv_0, Conv_1, Dense_0...)
    layer_names = sorted(params.keys())

    layers_found = []
    for name in layer_names:
        # Determine Type based on name
        if 'Conv' in name:
            # Inspect kernel shape: (k_h, k_w, in_channels, out_channels)
            kernel_shape = params[name]['kernel'].shape
            desc = f"{kernel_shape[0]}x{kernel_shape[1]} Filters\nCh: {kernel_shape[2]}->{kernel_shape[3]}"
            color = '#add8e6' # Blue
            l_type = "Conv"
        elif 'Dense' in name:
            # Inspect kernel shape: (in_features, out_features)
            kernel_shape = params[name]['kernel'].shape
            desc = f"Units: {kernel_shape[1]}"
            color = '#90ee90' # Green
            l_type = "Dense"
        else:
            continue

        layers_found.append({
            'name': name,
            'type': l_type,
            'desc': desc,
            'color': color
        })

    # 3. Plotting Logic
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.axis('off')

    n_layers = len(layers_found)
    box_width = 0.12
    box_height = 0.5
    spacing = 0.18
    start_x = 0.05
    y_center = 0.5

    print(f"Detected {n_layers} layers in {model_class.__name__}: {[l['name'] for l in layers_found]}")

    for i, layer in enumerate(layers_found):
        x = start_x + i * spacing

        # Draw Box
        rect = patches.Rectangle((x, y_center - box_height/2), box_width, box_height,
                                 linewidth=2, edgecolor='black', facecolor=layer['color'], zorder=2)
        ax.add_patch(rect)

        # Text: Layer Name
        ax.text(x + box_width/2, y_center + 0.15, layer['name'],
                ha='center', va='bottom', fontsize=11, weight='bold')

        # Text: Description (Dimensions)
        ax.text(x + box_width/2, y_center, layer['desc'],
                ha='center', va='center', fontsize=9)

        # Draw Arrow
        if i < n_layers - 1:
            arrow_start = x + box_width
            arrow_end = start_x + (i + 1) * spacing
            ax.arrow(arrow_start, y_center, arrow_end - arrow_start - 0.01, 0,
                     head_width=0.04, head_length=0.03, fc='k', ec='k', zorder=1)

            # Label implicit operations between layers
            if layer['type'] == 'Conv' and layers_found[i+1]['type'] == 'Conv':
                 ax.text((arrow_start + arrow_end)/2, y_center + 0.02, "ReLU\nPool",
                        ha='center', va='bottom', fontsize=8, color='gray')
            elif layer['type'] == 'Conv' and layers_found[i+1]['type'] == 'Dense':
                 ax.text((arrow_start + arrow_end)/2, y_center + 0.02, "Flatten",
                        ha='center', va='bottom', fontsize=8, color='gray')

    plt.title(f"Dynamic Architecture Visualization: {model_class.__name__}", fontsize=16)
    plt.tight_layout()
    plt.show()

# Run the visualizer
draw_cnn_architecture(CNN)

#Part 2: Task A - Training the Base Model

**Task**: Classify **Frogs** (Class 6) vs **Ships** (Class 8). **Dataset**: CIFAR-10.

We train this model from scratch. The network will learn to recognize edges, shapes, and textures associated with these objects.

In [ ]:
# --- Step 4: Setup Task A (Frogs vs Ships) ---

# CIFAR-10 Indices: 6=Frog, 8=Ship
TASK_A_CLASSES = [6, 8]
train_ds_A = get_dataset('cifar10', TASK_A_CLASSES, 'train[:20%]') # Use 20% for speed
test_ds_A = get_dataset('cifar10', TASK_A_CLASSES, 'test[:20%]')

# Visual Check
sample_batch = next(iter(train_ds_A))
show_batch(sample_batch, "Task A: 0=Frog, 1=Ship")

# --- Step 5: Training Utils ---

def create_train_state(rng, learning_rate=0.001):
    """Initialize the model parameters and optimizer state."""
    cnn = CNN()
    params = cnn.init(rng, jnp.ones([1, 32, 32, 3]))['params']
    tx = optax.adam(learning_rate)
    return train_state.TrainState.create(apply_fn=cnn.apply, params=params, tx=tx)

@jax.jit
def train_step(state, batch):
    """Perform a single training step."""
    images, labels = batch

    def loss_fn(params):
        logits = state.apply_fn({'params': params}, images)
        # Cross Entropy Loss
        one_hot = jax.nn.one_hot(labels, 2)
        loss = optax.softmax_cross_entropy(logits=logits, labels=one_hot).mean()
        return loss, logits

    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(state.params)
    state = state.apply_gradients(grads=grads)

    # Compute accuracy
    acc = jnp.mean(jnp.argmax(logits, -1) == labels)
    return state, loss, acc

# --- Step 6: Train Base Model ---

rng = random.PRNGKey(0)
state_A = create_train_state(rng)

print("Training Task A (Base Model)...")
loss_history_A = []

for epoch in range(10): # Train for 10 epochs
    batch_losses = []
    batch_accs = []
    for batch in train_ds_A:
        state_A, loss, acc = train_step(state_A, batch)
        batch_losses.append(loss)
        batch_accs.append(acc)

    avg_loss = np.mean(batch_losses)
    avg_acc = np.mean(batch_accs)
    loss_history_A.append(avg_loss)
    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}, Accuracy = {avg_acc:.4f}")

plt.plot(loss_history_A)
plt.title("Task A Learning Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()


## 4. Visualizing Intermediate "Receptors" (Feature Maps)

Before we move to Task B (Humans vs. Cars), it is crucial to understand **what** we are actually transferring.

In a Convolutional Neural Network (CNN), the early layers act as **feature extractors**. They don't know what a "Frog" or a "Ship" is; instead, they learn to detect low-level visual patterns like:

* **Edges** (Vertical, Horizontal, Diagonal)
* **Color gradients**
* **Simple textures** (bumps, smooth patches)

These low-level features are **universal**. An edge that defines the hull of a *Ship* is remarkably similar to an edge that defines the bumper of a *Car*. This is why Transfer Learning works: we reuse these pre-learned "receptors" instead of learning how to see edges from scratch.

### **Understanding the Code**

In the following cell, we will "peek" inside the trained model:

1. **Sub-Model Definition**: We define a mini-model (`FirstLayer`) that contains *only* the first Convolutional layer.
2. **Parameter Extraction**: We extract the trained weights (`Conv_0`) from our main `state_A` and load them into this mini-model.
3. **Forward Pass**: We push a single test image through this layer.
* Input: `(32, 32, 3)` — Height, Width, RGB Channels.
* Output: `(32, 32, 32)` — Height, Width, **32 Feature Channels**.


4. **Visualization**: We plot 15 of these 32 channels. Brighter regions indicate where that specific filter "fired" (found a match).

**What to look for:**

* Some filters might act as **edge detectors**, lighting up the outlines of the object.
* Others might react to specific **colors** (e.g., activating only on the blue sea or green frog).
* These are the "learned eyes" we will donate to Task B.

In [ ]:
# --- Step 6b: Visualize Intermediate "Receptors" ---
# To understand what we are transferring, let's visualize the Feature Maps
# (the output of the first Convolutional layer) for a specific test image.

def visualize_feature_maps(state, image):
    # 1. Define a mini-model that only runs the first layer
    # We essentially replicate the first part of our CNN class
    class FirstLayer(nn.Module):
        @nn.compact
        def __call__(self, x):
            x = nn.Conv(features=32, kernel_size=(3, 3), name='Conv_0')(x)
            x = nn.relu(x)
            return x

    # 2. Extract the params specifically for the first layer
    # The keys in state.params correspond to the names we gave (Conv_0, etc.)
    base_params = state.params
    layer_params = {'params': {'Conv_0': base_params['Conv_0']}}

    # 3. Run the image through this layer
    # We add a batch dimension [1, 32, 32, 3]
    input_img = image[None, ...]
    feature_maps = FirstLayer().apply(layer_params, input_img)

    # Remove batch dim -> [32, 32, 32]
    feature_maps = feature_maps[0]

    # 4. Plotting
    # We will plot the Original Image + 15 representative feature maps
    fig, axes = plt.subplots(4, 4, figsize=(10, 10))

    # Plot Original
    ax = axes[0, 0]
    ax.imshow(image)
    ax.set_title("Original Input")
    ax.axis('off')

    # Plot Activations
    # These show what the network "sees" after the first filter bank
    for i in range(15):
        row = (i + 1) // 4
        col = (i + 1) % 4
        ax = axes[row, col]

        # Plot the i-th channel of the feature map
        # We use a 'viridis' colormap to show activation intensity
        map_img = feature_maps[:, :, i]
        ax.imshow(map_img, cmap='viridis')
        ax.set_title(f"Filter {i}")
        ax.axis('off')

    plt.suptitle(f"Layer 1 Activations (Feature Receptors)", fontsize=16)
    plt.tight_layout()
    plt.show()

# Get a random image from the test set (e.g., a Ship)
test_batch = next(iter(test_ds_A))
test_img = test_batch[0][0] # First image of the batch

visualize_feature_maps(state_A, test_img)

#Part 3: Task B - Transfer Learning (Humans vs Cars)

**Task:** Classify **People** vs **Vehicles**. Dataset: CIFAR-100. **Method:** We will use the parameters (`state_A.params`) trained on frogs/ships.

1. **Freeze** the Convolutional layers (Feature Extractor).

2. **Reset** the Dense layers (Classifier Head).

3. **Train** only the Dense layers on the new data.

This simulates a scenario where we have limited data or compute for Task B and want to leverage the visual understanding the model gained from Task A.

In [ ]:
# --- Step 7: Setup Task B (Humans vs Cars) ---

# CIFAR-100 has "superclasses".
# We manually pick indices for "People" (e.g., boy, girl, man, woman)
# And "Vehicles" (e.g., bus, pickup_truck, train, streetcar)

# Class Indices in CIFAR-100:
# People: 11 (boy), 35 (girl), 46 (man), 98 (woman) -> Map to 0
# Cars/Vehicles: 8 (bicycle), 13 (bus), 48 (motorcycle), 58 (pickup_truck) -> Map to 1

# Let's simplify: 46 (Man) vs 58 (Pickup Truck)
TASK_B_CLASSES = [46, 58]
train_ds_B = get_dataset('cifar100', TASK_B_CLASSES, 'train[:20%]')
test_ds_B = get_dataset('cifar100', TASK_B_CLASSES, 'test[:20%]')

sample_batch_B = next(iter(train_ds_B))
show_batch(sample_batch_B, "Task B: 0=Man, 1=Truck")

# --- Step 8: Transfer Logic  ---

# 1. Start with the trained parameters from Task A
base_params = state_A.params

# 2. Re-initialize ONLY the dense layers (Head)
rng_B = random.PRNGKey(1)
fresh_params = CNN().init(rng_B, jnp.ones([1, 32, 32, 3]))['params']

# 3. Mix and Match: Keep Conv from A, take Dense from Fresh

# This function converts a FrozenDict to a mutable dict.
# If it is already a dict, it returns it as-is, preventing the AttributeError.
transfer_params = unfreeze(base_params)

# Now that it is a mutable dictionary, we can swap the layers
transfer_params['Dense_0'] = fresh_params['Dense_0']
transfer_params['Dense_1'] = fresh_params['Dense_1']

# (Optional) Verify structure
print("Transfer params keys:", transfer_params.keys())


# --- Step 9: Frozen Training Step  ---


# Create a new optimizer (reset momentum)
tx_B = optax.adam(0.001)
state_B = train_state.TrainState.create(apply_fn=CNN().apply, params=transfer_params, tx=tx_B)

@jax.jit
def train_step_frozen(state, batch):
    images, labels = batch

    def loss_fn(params):
        logits = state.apply_fn({'params': params}, images)
        one_hot = jax.nn.one_hot(labels, 2)
        return optax.softmax_cross_entropy(logits=logits, labels=one_hot).mean()

    grad_fn = jax.grad(loss_fn)
    grads = grad_fn(state.params)

    # --- THE MAGIC: Mask Gradients ---
    grads = unfreeze(grads)

    # We set gradients for Conv layers to 0 so they don't update.
    # We use jax.tree_util.tree_map to handle the nested weight/bias structure inside 'Conv_0'
    grads['Conv_0'] = jax.tree_util.tree_map(jnp.zeros_like, grads['Conv_0'])
    grads['Conv_1'] = jax.tree_util.tree_map(jnp.zeros_like, grads['Conv_1'])
    grads['Conv_2'] = jax.tree_util.tree_map(jnp.zeros_like, grads['Conv_2'])

    # Apply updates
    state = state.apply_gradients(grads=grads)
    return state, loss_fn(state.params)

# --- Step 10: Train Task B ---

print("\nTraining Task B (Transfer Learning - Head Only)...")
loss_history_B = []

for epoch in range(10):
    batch_losses = []
    for batch in train_ds_B:
        state_B, loss = train_step_frozen(state_B, batch)
        batch_losses.append(loss)

    avg_loss = np.mean(batch_losses)
    loss_history_B.append(avg_loss)
    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}")

plt.plot(loss_history_B, color='orange')
plt.title("Task B Learning Curve (Frozen Backbone)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

# --- Step 11: Final Evaluation ---
# Let's see if our Frog-detector knows how to detect Humans vs Cars!

total = 0
correct = 0
for batch in test_ds_B:
    imgs, lbls = batch
    logits = state_B.apply_fn({'params': state_B.params}, imgs)
    preds = jnp.argmax(logits, -1)
    correct += jnp.sum(preds == lbls)
    total += len(lbls)

print(f"\nFinal Accuracy on Humans vs Cars: {correct/total:.2%}")

# Conclusion
We successfully trained a classifier to distinguish Humans from Cars without the network ever learning "Human" features from scratch.

We trained filters to recognize **Frogs and Ships**.

Those filters (edges, curves, textures) were **transferred** to the new task.

We only optimized the final decision layer.

This demonstrates that lower-level visual features are **universal** across different visual domains.